# Mingrammer Diagrams Documentation Crawler

Crawls all documentation pages from the mingrammer/diagrams docs site via the sidebar menu, saves raw HTML, then converts to structured Markdown under `docs/mingrammer_diagrams_0.25.x/`.


## Library Imports


In [1]:
import os
import re
import time
import random
import json
import warnings
from urllib.parse import urljoin

import requests
from bs4 import BeautifulSoup
from tqdm import tqdm
from html_to_markdown import convert, ConversionOptions

warnings.filterwarnings("ignore")
print("All imports loaded successfully.")


All imports loaded successfully.


## Configuration

All tunable parameters are defined here. Directories are created automatically.


In [2]:
# Project identifier and URLs
NAME = "mingrammer-diagrams-0.25.x"
BASE_URL = "https://diagrams.mingrammer.com"
START_URL = "https://diagrams.mingrammer.com/docs/getting-started/installation"

# Output directories following docs/<topic>_v<version>/ convention
BASE_DIR = f"docs/{NAME.replace('-', '_')}"
HTML_DIR = f"{BASE_DIR}/html"
REFERENCES_DIR = f"{BASE_DIR}/references"
REFERENCES_JSON = os.path.join(BASE_DIR, "references.json")

# Throttle between requests to avoid rate-limiting
DELAY_RANGE = (0.2, 0.6)
# Skip re-downloading files newer than this threshold
MAX_AGE_DAYS = 7
MAX_AGE_SECONDS = MAX_AGE_DAYS * 86400

# Browser-like headers to avoid being blocked
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

# CSS selectors discovered during manual site inspection
SIDEBAR_SELECTOR = "div.navGroups"
CONTENT_SELECTOR = "div.post"

print("Configuration:")
print(f"  NAME              : {NAME}")
print(f"  BASE_URL          : {BASE_URL}")
print(f"  START_URL         : {START_URL}")
print(f"  BASE_DIR          : {BASE_DIR}")
print(f"  HTML_DIR          : {HTML_DIR}")
print(f"  REFERENCES_DIR    : {REFERENCES_DIR}")
print(f"  REFERENCES_JSON   : {REFERENCES_JSON}")
print(f"  DELAY_RANGE       : {DELAY_RANGE}")
print(f"  MAX_AGE_DAYS      : {MAX_AGE_DAYS}")
print(f"  SIDEBAR_SELECTOR  : {SIDEBAR_SELECTOR}")
print(f"  CONTENT_SELECTOR  : {CONTENT_SELECTOR}")

os.makedirs(HTML_DIR, exist_ok=True)
os.makedirs(REFERENCES_DIR, exist_ok=True)
print("Directories created.")


Configuration:
  NAME              : mingrammer-diagrams-0.25.x
  BASE_URL          : https://diagrams.mingrammer.com
  START_URL         : https://diagrams.mingrammer.com/docs/getting-started/installation
  BASE_DIR          : docs/mingrammer_diagrams_0.25.x
  HTML_DIR          : docs/mingrammer_diagrams_0.25.x/html
  REFERENCES_DIR    : docs/mingrammer_diagrams_0.25.x/references
  REFERENCES_JSON   : docs/mingrammer_diagrams_0.25.x\references.json
  DELAY_RANGE       : (0.2, 0.6)
  MAX_AGE_DAYS      : 7
  SIDEBAR_SELECTOR  : div.navGroups
  CONTENT_SELECTOR  : div.post
Directories created.


## Step 2: Fetch Start Page

Downloads the starting documentation page and parses it with BeautifulSoup.


In [3]:
print("Fetching start page...")

# Parse the start page to extract the sidebar navigation tree
response = requests.get(START_URL, headers=HEADERS)
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")
print(f"Start page fetched successfully ({len(response.text)} bytes).")


Fetching start page...
Start page fetched successfully (14544 bytes).


## Step 3: Extract Sidebar Menu Links

Finds the sidebar navigation container, extracts all internal doc links, deduplicates, and saves to `references.json` with `title`, `slug`, and `url`.


In [4]:
print("Scanning sidebar menu...")

# Locate the sidebar container using the pre-configured CSS selector
sidebar_container = soup.select_one(SIDEBAR_SELECTOR)

if sidebar_container is None:
    raise Exception(f"Sidebar container '{SIDEBAR_SELECTOR}' not found!")

# Extract all anchor tags from the sidebar
links = sidebar_container.find_all("a", href=True)

# Deduplicate URLs and filter only internal doc paths
entries = []
seen_urls = set()

for link in links:
    href = link["href"].strip()

    # Keep only doc pages; skip anchors and external links
    if not href.startswith("/docs/") or "#" in href:
        continue

    full_url = urljoin(BASE_URL, href)

    if full_url in seen_urls:
        continue
    seen_urls.add(full_url)

    title = link.get_text(strip=True)
    if not title:
        continue

    # Slugify menu name for safe filenames (lowercase, hyphens, no special chars)
    slug = title.lower().strip()
    slug = re.sub(r'[\/*?:"<>| ]', "-", slug)
    slug = re.sub(r"-+", "-", slug).strip("-")
    entries.append({"title": title, "slug": slug, "url": full_url})

# Assign sequential index and save to JSON for subsequent steps
for idx, entry in enumerate(entries, start=1):
    entry["idx"] = idx

with open(REFERENCES_JSON, "w", encoding="utf-8") as f:
    json.dump(entries, f, indent=2, ensure_ascii=False)

print(f"Total menu items found: {len(entries)}")
print(f"Saved to: {REFERENCES_JSON}")
print(f"\nFirst 5 items:")
for entry in entries[:5]:
    print(f"  {entry['idx']:03d} - {entry['title']} -> {entry['url']}")


Scanning sidebar menu...
Total menu items found: 25
Saved to: docs/mingrammer_diagrams_0.25.x\references.json

First 5 items:
  001 - Installation -> https://diagrams.mingrammer.com/docs/getting-started/installation
  002 - Examples -> https://diagrams.mingrammer.com/docs/getting-started/examples
  003 - Diagrams -> https://diagrams.mingrammer.com/docs/guides/diagram
  004 - Nodes -> https://diagrams.mingrammer.com/docs/guides/node
  005 - Clusters -> https://diagrams.mingrammer.com/docs/guides/cluster


## Step 4: Crawl Each Page

Reads `references.json`, downloads each page, and saves HTML to `HTML_DIR` with 3-digit zero-padded prefixes. Skips existing files that are still within `MAX_AGE_DAYS`.


In [5]:
print("Starting crawling process...")

# Read the link metadata saved by Step 3
with open(REFERENCES_JSON, "r", encoding="utf-8") as f:
    entries = json.load(f)

errors = []
skipped = 0

for entry in tqdm(entries, desc="Crawling"):
    # Build file path with zero-padded prefix for consistent ordering
    filename = f"{entry['idx']:03d}_{entry['slug']}.html"
    file_path = os.path.join(HTML_DIR, filename)

    # Skip download if file exists and is still within cache age
    if os.path.exists(file_path):
        file_age = time.time() - os.path.getmtime(file_path)
        if file_age < MAX_AGE_SECONDS:
            skipped += 1
            continue

    try:
        r = requests.get(entry["url"], headers=HEADERS)
        r.raise_for_status()

        with open(file_path, "w", encoding="utf-8") as f:
            f.write(r.text)

        # Polite delay between requests to avoid overwhelming the server
        time.sleep(random.uniform(*DELAY_RANGE))

    except Exception as e:
        errors.append(f"Failed {entry['url']}: {e}")

if skipped:
    print(f"\nSkipped {skipped} files (cached within {MAX_AGE_DAYS} days).")
if errors:
    print(f"\nErrors encountered ({len(errors)}):")
    for err in errors:
        print(f"  {err}")

print(f"\nCrawling finished. Files saved in: {HTML_DIR}")


Starting crawling process...


Crawling: 100%|██████████| 25/25 [00:00<00:00, 4995.36it/s]


Skipped 25 files (cached within 7 days).

Crawling finished. Files saved in: docs/mingrammer_diagrams_0.25.x/html


## Step 5: Convert HTML to Markdown

Reads `references.json`, converts each HTML file to Markdown, and saves individual `.md` files to `REFERENCES_DIR`.


In [6]:
# Read the link metadata saved by Step 3
with open(REFERENCES_JSON, "r", encoding="utf-8") as f:
    entries = json.load(f)

print(f"Total entries to convert: {len(entries)}")

# Use ATX headings (#) and disable text wrapping for cleaner output
options = ConversionOptions(heading_style="atx", wrap=False)
conversion_errors = []

for entry in tqdm(entries, desc="Converting"):
    html_filename = f"{entry['idx']:03d}_{entry['slug']}.html"
    html_path = os.path.join(HTML_DIR, html_filename)

    with open(html_path, "r", encoding="utf-8") as f:
        html_content = f.read()

    soup = BeautifulSoup(html_content, "html.parser")

    # Isolate the main content area using the known selector
    content_div = soup.select_one(CONTENT_SELECTOR)

    if content_div is None:
        conversion_errors.append(f"Content container not found in: {html_filename}")
        continue

    # Strip image-related tags to keep output lean
    for tag in ["img", "svg", "picture", "source"]:
        for element in content_div.find_all(tag):
            element.decompose()

    result = convert(str(content_div), options)

    # Mirror the HTML filename as a .md file
    md_filename = html_filename.replace(".html", ".md")
    md_file_path = os.path.join(REFERENCES_DIR, md_filename)

    with open(md_file_path, "w", encoding="utf-8") as f:
        # Library returns ConversionResult in some versions, plain str in others
        markdown_text = result.content if hasattr(result, "content") else result
        f.write(markdown_text)

if conversion_errors:
    print(f"\nConversion errors ({len(conversion_errors)}):")
    for err in conversion_errors:
        print(f"  {err}")

print(f"\nConversion completed. Files saved in: {REFERENCES_DIR}")


Total entries to convert: 25


Converting: 100%|██████████| 25/25 [00:02<00:00,  9.87it/s]


Conversion completed. Files saved in: docs/mingrammer_diagrams_0.25.x/references


## Step 6: Generate references.md Index

Reads `references.json` and generates a basic `references.md` index file with original titles and links.


In [7]:
REFERENCES_INDEX = os.path.join(BASE_DIR, "references.md")

# Read the link metadata saved by Step 3
with open(REFERENCES_JSON, "r", encoding="utf-8") as f:
    entries = json.load(f)

print(f"Generating references.md index with {len(entries)} entries...")

# Build a simple bullet list using the original sidebar titles
lines = []
lines.append("# Mingrammer Diagrams (0.25.x) Documentation Reference")
lines.append("")
lines.append(f"Crawled from [{BASE_URL}]({START_URL})")
lines.append("")
lines.append("## Contents")
lines.append("")

for entry in entries:
    link_target = f"references/{entry['idx']:03d}_{entry['slug']}.md"
    lines.append(f"- **{entry['idx']:03d}** - [{entry['title']}]({link_target})")

lines.append("")
lines.append("---")
lines.append("")
lines.append("> **Note:** This file was auto-generated. An AI should refine it with descriptions, section groupings, and context.")

content = "\n".join(lines)

with open(REFERENCES_INDEX, "w", encoding="utf-8") as f:
    f.write(content)

print(f"references.md generated at: {REFERENCES_INDEX}")


Generating references.md index with 25 entries...
references.md generated at: docs/mingrammer_diagrams_0.25.x\references.md


## Summary

Crawling complete. Output structure:

- `docs/mingrammer_diagrams_0.25.x/references.json` — Link metadata (title, slug, url)
- `docs/mingrammer_diagrams_0.25.x/references.md` — Auto-generated index file (AI should refine)
- `docs/mingrammer_diagrams_0.25.x/html/` — Raw HTML pages
- `docs/mingrammer_diagrams_0.25.x/references/` — Individual Markdown files

Next step: AI refines `references.md` with descriptions, section groupings, and context.
